<a href="https://colab.research.google.com/github/advayc/tencent-auk/blob/main/tencent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tencent AuK speech editing

This notebook uses Tencent AuK to edit an existing speech recording.

It supports:
- Transcript/content editing: replace, insert, or remove words while preserving the speaker and recording style.
- Tone/emotion editing: change emotion such as calm, happy, sad, surprised, or excited.
- Speech cleanup: remove background noise and/or room reverb.
- Speaker isolation: keep one speaker by speaking order or by something they said.

AuK exposes these tasks through a natural-language instruction interface. This notebook uses AuK-Flash by default because it uses the official 4-step distilled model. You can switch to the base model in the configuration cell if you have enough GPU memory.

**Recommended Colab runtime:** GPU. An L4/A100 is preferable for longer recordings. CPU offload is enabled by default to reduce VRAM usage.


In [1]:
# Check the Colab runtime before installing AuK.

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No CUDA GPU is available. Inference will be impractical on CPU for most recordings.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM GB: 14.56


## 1. Install AuK

The official AuK package currently targets the PyTorch 2.7 ABI. The install below lets AuK install its compatible dependencies.


In [1]:
!git clone -q https://github.com/Tencent-Hunyuan/AuK.git
%cd /content/AuK

%pip uninstall -y torch torchvision torchaudio
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 \
  --index-url https://download.pytorch.org/whl/cu128

%pip install -q -e .

/content/AuK
Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
Found existing installation: torchaudio 2.11.0+cpu
Uninstalling torchaudio-2.11.0+cpu:
  Successfully uninstalled torchaudio-2.11.0+cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 120.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 325.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 135.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 128.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 124.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 103.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 116.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

## 2. Choose the model

`AuK-Flash` is the default. It is the official distilled 4-step version.

Set `USE_FLASH = False` if you want the base AuK checkpoint instead. The base model is slower but gives you configurable sampling steps.

`CPU_OFFLOAD = True` moves the large Qwen encoder and diffusion transformer between CPU and GPU during inference. This lowers VRAM usage but makes inference slower.


In [3]:
USE_FLASH = True
CPU_OFFLOAD = True
DTYPE = "bf16"

MODEL_DIR = "/content/AuK/ckpts"
AUK_DIR = f"{MODEL_DIR}/AuK"
FLASH_DIR = f"{MODEL_DIR}/AuK-Flash"
QWEN_DIR = f"{MODEL_DIR}/Qwen2.5-Omni-3B"

CHECKPOINT_DIR = FLASH_DIR if USE_FLASH else AUK_DIR
CHECKPOINT = f"{CHECKPOINT_DIR}/auk_flash.safetensors" if USE_FLASH else f"{CHECKPOINT_DIR}/auk_base.safetensors"
CONFIG = f"{CHECKPOINT_DIR}/config.yaml"

print("Using:", "AuK-Flash" if USE_FLASH else "AuK Base")
print("CPU offload:", CPU_OFFLOAD)


Using: AuK-Flash
CPU offload: True


## 3. Download the model weights

This downloads the selected AuK checkpoint plus Qwen2.5-Omni-3B, which AuK uses as its multimodal encoder. The downloads can take several minutes and require substantial disk space.


In [4]:
import os

os.makedirs(MODEL_DIR, exist_ok=True)

if USE_FLASH:
    !hf download tencent/AuK-Flash --local-dir /content/AuK/ckpts/AuK-Flash
else:
    !hf download tencent/AuK --local-dir /content/AuK/ckpts/AuK

!hf download Qwen/Qwen2.5-Omni-3B --local-dir /content/AuK/ckpts/Qwen2.5-Omni-3B

print("Model download complete.")


Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0% 0/8 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Still waiting to acquire lock on /content/AuK/ckpts/AuK-Flash/.cache/huggingface/.gitignore.lock (elapsed: 0.1 seconds)

Reconstructing (incomplete total...):   0% 0.00/1.65k [00:00<?, ?B/s]         

Fetching 8 files:  12% 1/8 [00:00<00:00,  7.81it/s]
Reconstructing (incomplete total...):   0% 1.65k/637M [00:00<13:05:06, 13.5kB/s]
Reconstructing (incomplete total...):   0% 1.65k/6.76G [00:00<138:46:53, 13.5kB/s]
Reconstructing (incomplete total...):   0% 1.65k/6.76G [00:00<138:49:40, 13.5kB/s]
Reconstructing (incomplete total...):   0% 1.65k/6.76G [00:00<138:50:43, 13.5kB/s]
Reconstructing (incomplete total...):   0

## 4. Load AuK

This initializes the model once. After this cell finishes, the editing functions below can reuse the same model without reloading it.


In [5]:
import os
import gc
import math
import torch
import torchaudio
from IPython.display import Audio, display

from auk.infer.infer_auk import AukInfer, save_audio

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Runtime > Change runtime type before loading AuK.")

engine = AukInfer(
    CONFIG,
    CHECKPOINT,
    device="cuda:0",
    dtype=DTYPE,
    qwen_path=QWEN_DIR,
    cpu_offload=CPU_OFFLOAD,
)

print("AuK is ready.")


OSError: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchaudio/lib/libtorchaudio.so

## 5. Upload an audio file

Use WAV, MP3, M4A, FLAC, or another format that your Colab audio backend can decode. Shorter clips are recommended for testing first.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file was uploaded.")

INPUT_AUDIO = "/content/" + next(iter(uploaded.keys()))
OUTPUT_DIR = "/content/AuK/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

audio_info = torchaudio.info(INPUT_AUDIO)
INPUT_DURATION = audio_info.num_frames / audio_info.sample_rate

print("Input:", INPUT_AUDIO)
print("Duration:", round(INPUT_DURATION, 2), "seconds")
print("Sample rate:", audio_info.sample_rate)
print("Channels:", audio_info.num_channels)

display(Audio(INPUT_AUDIO))


## 6. Core helper

All four workflows use the same helper. For content editing, AuK needs a target duration because changing words can change the output length. For emotion, enhancement, and speaker separation, the official cookbook uses same-length output, so `gen_seconds` is omitted.


In [ ]:
def run_auk_edit(instruction, output_name, audio_path=INPUT_AUDIO, gen_seconds=None):
    output_path = os.path.join(OUTPUT_DIR, output_name)

    content = [{"type": "text", "text": instruction}]
    if audio_path is not None:
        content.append({"type": "audio", "audio": audio_path})

    messages = [{"role": "user", "content": content}]

    print("Instruction:", instruction)
    print("Running AuK...")

    audio, sr = engine.generate(
        messages,
        gen_seconds=gen_seconds,
    )

    save_audio(audio, sr, output_path)
    print("Saved:", output_path)
    display(Audio(output_path))

    return output_path


# 7. Transcript / content editing

Replace words or phrases in an existing recording while keeping the voice and surrounding speech natural.

Examples:
- `Replace 'Tuesday' with 'Thursday'.`
- `Replace 'the old system' with 'the new system'.`
- `Add 'actually' before 'we tested it'.`
- `Remove 'basically'.`

For a first test, keep the replacement reasonably close to the original in length.


In [ ]:
ORIGINAL_TEXT = "Tuesday"
NEW_TEXT = "Thursday"

# You can increase or decrease this if the replacement is much shorter or longer.
TARGET_SECONDS = round(INPUT_DURATION, 2)

instruction = f"Replace '{ORIGINAL_TEXT}' with '{NEW_TEXT}'."

content_output = run_auk_edit(
    instruction,
    "content_edit.wav",
    gen_seconds=TARGET_SECONDS,
)


# 8. Tone / emotion editing

AuK can change the emotion while preserving the spoken content and speaker. Supported cookbook examples include `happy`, `angry`, `sad`, `fearful`, `surprised`, `disgusted`, `calm`, and `excited`.


In [ ]:
EMOTION = "excited"

instruction = f"Change the emotion to {EMOTION}."

emotion_output = run_auk_edit(
    instruction,
    "emotion_edit.wav",
)


# 9. Speech cleanup

This uses AuK's speech enhancement task. You can remove only background noise, remove room reverb, or remove both while preserving the speakers.


In [ ]:
CLEANUP_MODE = "noise_and_reverb"

if CLEANUP_MODE == "noise_only":
    instruction = "Remove only the background noise, preserve everything else, and output audio of the same length."
elif CLEANUP_MODE == "reverb_only":
    instruction = "Remove only the room reverberation, preserve everything else, and output audio of the same length."
elif CLEANUP_MODE == "noise_and_reverb":
    instruction = "Preserve all speakers, remove noise and reverberation, and output clean speech of the same length."
else:
    raise ValueError("CLEANUP_MODE must be noise_only, reverb_only, or noise_and_reverb")

cleanup_output = run_auk_edit(
    instruction,
    "speech_cleanup.wav",
)


# 10. Speaker isolation by speaking order

If multiple people are talking, AuK can keep the first speaker to start talking, second speaker to start talking, and so on.

Set `SPEAKER_NUMBER = 1` for the first speaker, `2` for the second, etc.


In [ ]:
SPEAKER_NUMBER = 2

def ordinal(n):
    if 10 <= n % 100 <= 20:
        suffix = "th"
    else:
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"

instruction = (
    f"Please keep the {ordinal(SPEAKER_NUMBER)} speaker to start talking "
    "and remove the other speakers, outputting a single clean speech track."
)

speaker_output = run_auk_edit(
    instruction,
    "speaker_isolated.wav",
)


# 11. Speaker isolation by transcript/content

If you know something the target speaker says, you can ask AuK to locate that speaker and remove the others.

Example: if one person says `I think we should ship it today`, use that phrase as the target.


In [ ]:
TARGET_SPEECH = "I think we should ship it today"

instruction = (
    f'Keep only the speaker who says "{TARGET_SPEECH}" and remove all other speakers.'
)

target_speaker_output = run_auk_edit(
    instruction,
    "target_speaker.wav",
)


# 12. Generic AuK editor

You can also write your own instruction. This is useful for combinations such as:
- changing pitch
- changing speaking speed
- removing breaths or coughs
- converting normal speech to a whisper
- changing timbre
- de-accenting speech

The instruction should clearly state what you want changed and what should be preserved.


In [ ]:
CUSTOM_INSTRUCTION = "Raise the pitch by 2 semitones."
CUSTOM_OUTPUT_NAME = "custom_edit.wav"

# Use gen_seconds=None for same-length tasks.
# For content or speed edits, set an explicit target duration.
CUSTOM_GEN_SECONDS = None

custom_output = run_auk_edit(
    CUSTOM_INSTRUCTION,
    CUSTOM_OUTPUT_NAME,
    gen_seconds=CUSTOM_GEN_SECONDS,
)


# 13. Download all generated files

This creates a ZIP containing the outputs from the current Colab session.


In [ ]:
import shutil
from google.colab import files

zip_path = "/content/AuK_speech_outputs"
archive_path = shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

print("Created:", archive_path)
files.download(archive_path)


## Notes

- AuK's official Python API accepts an instruction and optional source audio, then returns generated audio.
- Content editing uses an explicit target duration in the official examples. The notebook defaults this to the original duration, but you can change it.
- Emotion editing, speech enhancement, and speaker separation are same-length tasks in the official cookbook.
- Longer recordings use more memory and take longer. Test with a short clip first.
- Only edit audio you have permission to modify and distribute.
